# Hybrid Quantum–Spiking LeNet-5 on MNIST, v2: ZZ-feature-map + strongly-entangling VQC

Same rate-encoded spiking LeNet-5 pipeline as v1; the **quantum F6 layer is a different VQC**:

```
 image ──rate code──► spikes (T steps)
   C1 conv5×5(6) → LIF → maxpool          6×14×14   ┐
   C3 conv5×5(16) → LIF → maxpool        16×5×5     │ classical spiking
   C5 FC 400→120 → LIF                    120       ┘
   Q  FC 120→n → a=π·σ(x) → ZZ feature map → [Rot ⊗ CNOT(range r_l)] ×L → ⟨Z_i⟩, ⟨Z_iZ_{i+1}⟩   2n   ← quantum F6
   F7 FC 2n→10 → LIF → spike count over T → class
```

| | v1 VQC | **v2 VQC (this notebook)** |
|---|---|---|
| Encoding | RY(π tanh x) angle encoding | **H⊗n + ZZ feature map** (Havlíček et al. 2019): RZ(a_i), IsingZZ((π−a_i)(π−a_j)) on ring |
| Variational block | RZ·RY per qubit | **Rot(φ,θ,ω)** = RZ(ω)RY(θ)RZ(φ) per qubit (full SU(2)) |
| Entangler | nearest-neighbour CNOT ring | **CNOT(i → i+r_l)**, range r_l = (l mod (n−1))+1 changes per layer (StronglyEntanglingLayers) |
| Re-uploading | RY before each layer | H + ZZ map before each layer |
| Readout | n × ⟨Z_i⟩ | **n × ⟨Z_i⟩ + n × ⟨Z_iZ_{i+1}⟩ = 2n features** (two-body correlators) |
| Params (n=6, L=3) | 36 | **54** |

$$|\psi(\mathbf a)\rangle=\prod_{l=1}^{L}\Big[\mathrm{Ent}_{r_l}\bigotimes_i \mathrm{Rot}(\phi_{li},\theta_{li},\omega_{li})\,U_{ZZ}(\mathbf a)\,H^{\otimes n}\Big]|0\rangle^{\otimes n},\quad
U_{ZZ}(\mathbf a)=\exp\!\Big(-\tfrac{i}{2}\Big[\sum_i a_i Z_i+\sum_{\langle ij\rangle}(\pi-a_i)(\pi-a_j)Z_iZ_j\Big]\Big)$$

The ZZ map is diagonal, so it is applied as one batched phase vector per sample. Simulation is exact, batched and differentiable (PyTorch), and validated against PennyLane in section 6b.

## 1. Setup

In [ ]:
# !pip install torch torchvision matplotlib scikit-learn tqdm pennylane   # pennylane optional (cross-check only)
import time, random, math, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 2. Configuration

In [ ]:
CFG = dict(
    # spiking
    T          = 25,       # simulation time steps
    gain       = 1.0,      # rate-encoding gain
    beta       = 0.9,      # LIF leak
    threshold  = 1.0,      # LIF threshold
    slope      = 25.0,     # surrogate-gradient steepness
    # quantum
    n_qubits   = 6,        # width of quantum layer
    q_layers   = 3,        # strongly-entangling layers
    reupload   = True,     # re-apply H + ZZ feature map before every layer
    shots      = None,     # None = exact expectations; int = finite-shot estimate at eval time
    # training
    epochs     = 10,
    batch_size = 128,
    lr         = 1e-3,     # classical parameters
    q_lr       = 5e-3,     # quantum circuit parameters
    seed       = 42,
    data_root  = "./data",
    ckpt       = "best_hybrid_qspiking_lenet_v2_zz_mnist.pt",
    run_baseline   = True, # also train classical spiking LeNet-5 for comparison (section 15)
    baseline_ckpt  = "best_spiking_lenet_mnist_baseline.pt",
)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
CFG

## 3. MNIST data
Pixels kept in [0,1] (no normalisation) so they serve directly as spike probabilities.

In [ ]:
tf = transforms.ToTensor()
train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=tf)

pin = torch.cuda.is_available()
train_dl = DataLoader(train_ds, CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=pin, drop_last=True)
test_dl  = DataLoader(test_ds,  CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}  batches/epoch: {len(train_dl)}")

## 4. Rate encoder

In [ ]:
def rate_encode(x, T, gain=1.0):
    """(B,C,H,W) in [0,1] -> (T,B,C,H,W) Bernoulli spikes with p = gain*x."""
    p = (x * gain).clamp(0.0, 1.0)
    return torch.bernoulli(p.unsqueeze(0).expand(T, *p.shape))

img, lbl = test_ds[0]
spk = rate_encode(img.unsqueeze(0), CFG["T"], CFG["gain"])[:, 0, 0]
fig, ax = plt.subplots(1, 6, figsize=(14, 2.6))
ax[0].imshow(img[0], cmap="gray"); ax[0].set_title(f"input (label {lbl})")
for i, t in enumerate([0, 1, 2, 3]):
    ax[i+1].imshow(spk[t], cmap="gray"); ax[i+1].set_title(f"spikes t={t}")
ax[5].imshow(spk.mean(0), cmap="gray"); ax[5].set_title(f"mean rate, T={CFG['T']}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Surrogate spike function and LIF neuron

In [ ]:
class FastSigmoidSpike(torch.autograd.Function):
    slope = CFG["slope"]
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g / (FastSigmoidSpike.slope * x.abs() + 1.0) ** 2

spike_fn = FastSigmoidSpike.apply


class LIF(nn.Module):
    def __init__(self, beta=0.9, threshold=1.0):
        super().__init__()
        self.beta, self.threshold, self.mem = beta, threshold, None
    def reset_state(self):
        self.mem = None
    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape:
            self.mem = torch.zeros_like(x)
        self.mem = self.beta * self.mem + x
        spk = spike_fn(self.mem - self.threshold)
        self.mem = self.mem - spk * self.threshold      # soft reset
        return spk

## 6. Quantum layer v2: ZZ feature map + strongly-entangling layers

Batched statevector simulator, shape `(B, 2**n)`, wire 0 = most significant bit (PennyLane convention).
* **H⊗n**: per-wire 2×2 contraction (on $|0\rangle^{\otimes n}$ it is just the uniform state).
* **ZZ feature map**: diagonal, so it is computed as one phase vector $e^{-\frac{i}{2}E_k(\mathbf a)}$ with $E_k=\sum_i a_i z_i(k)+\sum_{\langle ij\rangle}\phi_{ij}z_i(k)z_j(k)$, as a single matmul.
* **Rot**: per-wire 2×2 contraction with $R_Z(\omega)R_Y(\theta)R_Z(\phi)$.
* **CNOT(i→i+r)** for all i: precomputed basis permutation per layer.
* Readout: $\langle Z_i\rangle$ and $\langle Z_iZ_{i+1}\rangle$ from $|\psi_k|^2$ via one matmul.

In [ ]:
def apply_1q(state, U, wire, n):
    """state (B,2**n); U (2,2) shared or (B,2,2) per-sample."""
    B = state.shape[0]
    s = state.view(B, 2 ** wire, 2, 2 ** (n - wire - 1))
    s = torch.einsum("ij,bajc->baic", U, s) if U.dim() == 2 else torch.einsum("bij,bajc->baic", U, s)
    return s.reshape(B, 2 ** n)

def rot_matrix(phi, theta, omega):
    """PennyLane Rot(phi,theta,omega) = RZ(omega) RY(theta) RZ(phi); inputs (n,) -> (n,2,2) complex."""
    c, s = torch.cos(theta / 2), torch.sin(theta / 2)
    ep = torch.exp(-0.5j * (phi + omega).to(torch.complex64))   # e^{-i(φ+ω)/2}
    em = torch.exp(-0.5j * (phi - omega).to(torch.complex64))   # e^{-i(φ-ω)/2}
    return torch.stack([torch.stack([ep * c, -em.conj() * s], -1),
                        torch.stack([em * s,  ep.conj() * c], -1)], -2)

H_GATE = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex64) / math.sqrt(2)


class ZZStronglyEntanglingVQC(nn.Module):
    """ZZ-feature-map encoding + StronglyEntanglingLayers ansatz, readout <Z_i> and <Z_i Z_{i+1}>.
    input x (B, n) -> output (B, 2n) in [-1, 1]."""

    def __init__(self, n_qubits=6, n_layers=3, reupload=True, shots=None):
        super().__init__()
        assert n_qubits >= 3, "ring ZZ map needs n >= 3"
        n, dim = n_qubits, 2 ** n_qubits
        self.n, self.L, self.reupload, self.shots = n, n_layers, reupload, shots
        self.weights = nn.Parameter(0.1 * math.pi * torch.randn(n_layers, n, 3))   # (phi, theta, omega)
        self.ranges = [(l % (n - 1)) + 1 for l in range(n_layers)]
        self.pairs = [(i, (i + 1) % n) for i in range(n)]                           # ring

        bits = torch.tensor([[(k >> (n - 1 - i)) & 1 for i in range(n)] for k in range(dim)])
        zs = 1.0 - 2.0 * bits.float()                                                # (dim, n): z_i(k) = ±1
        zz = torch.stack([zs[:, i] * zs[:, j] for i, j in self.pairs], 1)           # (dim, n)
        self.register_buffer("zsign", zs)
        self.register_buffer("zzsign", zz)
        self.register_buffer("H", H_GATE.clone())

        # CNOT(i -> i+r) for i = 0..n-1 applied in order, as one basis permutation per layer
        def ent(b, r):
            b = list(b)
            for i in range(n):
                if b[i]: b[(i + r) % n] ^= 1
            return b
        to_int = lambda b: int("".join(map(str, b)), 2)
        for l, r in enumerate(self.ranges):
            f = torch.tensor([to_int(ent(b.tolist(), r)) for b in bits])
            self.register_buffer(f"perm{l}", torch.argsort(f))

    def feature_phase(self, a):
        """Diagonal ZZ feature map: (B,n) angles -> (B,dim) complex phases."""
        phi = torch.stack([(math.pi - a[:, i]) * (math.pi - a[:, j]) for i, j in self.pairs], 1)   # (B,n)
        E = a @ self.zsign.T + phi @ self.zzsign.T                                               # (B,dim)
        return torch.exp(-0.5j * E.to(torch.complex64))

    def forward(self, x):
        B, n, dim = x.shape[0], self.n, 2 ** self.n
        a = math.pi * torch.sigmoid(x)                         # angles in (0, π)
        phase = self.feature_phase(a)
        state = torch.full((B, dim), dim ** -0.5, dtype=torch.complex64, device=x.device)  # H⊗n|0>
        for l in range(self.L):
            if l == 0 or self.reupload:
                if l > 0:
                    for w in range(n): state = apply_1q(state, self.H, w, n)
                state = state * phase                          # ZZ feature map
            U = rot_matrix(self.weights[l, :, 0], self.weights[l, :, 1], self.weights[l, :, 2])
            for w in range(n): state = apply_1q(state, U[w], w, n)
            state = state[:, getattr(self, f"perm{l}")]        # CNOT(i -> i+r_l)
        probs = state.real ** 2 + state.imag ** 2
        out = torch.cat([probs @ self.zsign, probs @ self.zzsign], 1)   # (B, 2n)
        if self.shots and not self.training:                   # finite-shot emulation (each obs is ±1-valued)
            p1 = ((1 - out) / 2).clamp(0, 1)
            out = 1 - 2 * torch.distributions.Binomial(self.shots, p1).sample() / self.shots
        return out

    @property
    def out_features(self):
        return 2 * self.n

    def extra_repr(self):
        return (f"n_qubits={self.n}, layers={self.L}, cnot_ranges={self.ranges}, reupload={self.reupload}, "
                f"shots={self.shots}, readout=<Z>x{self.n} + <ZZ>x{self.n}")

### 6a. Circuit schematic (text, no dependencies)

In [ ]:
def draw_circuit_v2(n, L, reupload):
    ranges = [(l % (n - 1)) + 1 for l in range(L)]
    rows = [f"q{i}: |0>─H─" for i in range(n)]
    for l in range(L):
        for i in range(n):
            enc = ("" if l == 0 else "H─") + f"[ZZmap x]─" if (l == 0 or reupload) else ""
            rows[i] += enc + f"Rot(w{l}{i})─"
        rows = [r + f"[CNOT i→i+{ranges[l]}]─" for r in rows]
    for i in range(n): rows[i] += f"⟨Z{i}⟩, ⟨Z{i}Z{(i+1)%n}⟩"
    print("\n".join(rows))
    print(f"\nZZ map = RZ(a_i) on each wire + IsingZZ((π-a_i)(π-a_j)) on ring pairs; CNOT ranges per layer: {ranges}")
    print(f"trainable circuit params: {3 * n * L}   readout features: {2 * n}   Hilbert dim: {2 ** n}")
draw_circuit_v2(CFG["n_qubits"], CFG["q_layers"], CFG["reupload"])

### 6b. Validation of the simulator against PennyLane (optional)
Outputs and gradients compared on random inputs. Skipped if PennyLane is not installed.

In [ ]:
try:
    import pennylane as qml

    def make_qnode(vqc, interface="torch"):
        n, L = vqc.n, vqc.L
        dev = qml.device("default.qubit", wires=n)
        @qml.qnode(dev, interface=interface)
        def circuit(x, weights):
            a = np.pi / (1 + np.exp(-x)) if not torch.is_tensor(x) else math.pi * torch.sigmoid(x)
            for l in range(L):
                if l == 0 or vqc.reupload:
                    for i in range(n): qml.Hadamard(wires=i)
                    for i in range(n): qml.RZ(a[i], wires=i)
                    for i, j in vqc.pairs: qml.IsingZZ((np.pi - a[i]) * (np.pi - a[j]), wires=[i, j])
                for i in range(n): qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
                for i in range(n): qml.CNOT(wires=[i, (i + vqc.ranges[l]) % n])
            return ([qml.expval(qml.PauliZ(i)) for i in range(n)] +
                    [qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)) for i, j in vqc.pairs])
        return circuit

    tv = ZZStronglyEntanglingVQC(4, 3)
    qn = make_qnode(tv)
    x = torch.randn(5, 4, dtype=torch.float64)
    ours = tv(x.float()).detach()
    ref = torch.stack([torch.stack(qn(xi, tv.weights.detach().double())) for xi in x]).float()
    print("max |ours - pennylane| (outputs)  =", (ours - ref).abs().max().item())

    w = tv.weights.detach().double().clone().requires_grad_(True)
    torch.stack(qn(x[0], w)).sum().backward()
    tv.zero_grad(); tv(x[:1].float()).sum().backward()
    print("max |grad diff| (weights)        =", (tv.weights.grad - w.grad.float()).abs().max().item())
except ImportError:
    print("PennyLane not installed - cross-check skipped.")

## 7. Hybrid Quantum–Spiking LeNet-5 (v2 VQC)

In [ ]:
class HybridQSpikingLeNet5(nn.Module):
    LAYERS = ["C1", "C3", "C5", "F7-out"]

    def __init__(self, n_qubits=6, q_layers=3, reupload=True, shots=None,
                 num_classes=10, beta=0.9, threshold=1.0):
        super().__init__()
        self.conv1  = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2  = nn.Conv2d(6, 16, 5)
        self.fc1    = nn.Linear(16 * 5 * 5, 120)
        self.q_in   = nn.Linear(120, n_qubits)                                    # classical -> qubit angles
        self.vqc    = ZZStronglyEntanglingVQC(n_qubits, q_layers, reupload, shots)  # quantum F6 (v2)
        self.fc_out = nn.Linear(2 * n_qubits, num_classes)                        # <Z> + <ZZ> -> classes
        self.lifs   = nn.ModuleList([LIF(beta, threshold) for _ in range(4)])

    def reset_state(self):
        for l in self.lifs: l.reset_state()

    def step(self, x):
        l1, l2, l3, lo = self.lifs
        s1 = F.max_pool2d(l1(self.conv1(x)), 2)
        s2 = F.max_pool2d(l2(self.conv2(s1)), 2)
        s3 = l3(self.fc1(s2.flatten(1)))
        q  = self.vqc(self.q_in(s3))                 # (B, 2n) in [-1,1]
        so = lo(self.fc_out(q))
        return so, (s1, s2, s3, so), q

    def forward(self, spikes_in, record=False):
        self.reset_state()
        T = spikes_in.shape[0]
        counts, rates, out_train, q_trace = 0.0, [0.0] * 4, [], []
        for t in range(T):
            so, layers, q = self.step(spikes_in[t])
            counts = counts + so
            if record:
                out_train.append(so.detach()); q_trace.append(q.detach())
                for i, s in enumerate(layers):
                    rates[i] += s.detach().mean().item() / T
        if record:
            return counts, rates, torch.stack(out_train), torch.stack(q_trace)
        return counts


model = HybridQSpikingLeNet5(CFG["n_qubits"], CFG["q_layers"], CFG["reupload"], CFG["shots"],
                             beta=CFG["beta"], threshold=CFG["threshold"]).to(device)
n_q = sum(p.numel() for p in model.vqc.parameters())
n_c = sum(p.numel() for p in model.parameters()) - n_q
print(model)
print(f"Classical params: {n_c:,} | Quantum params: {n_q} | Total: {n_c + n_q:,}")

### 7a. Print the VQC layer
Module summary, trainable weights, and the full circuit (text and figure) drawn with PennyLane, using the real angles of one test digit at t=0. Re-run after training to see the trained weights. The last lines check that the PennyLane circuit reproduces the model's quantum output.

In [ ]:
# !pip install pennylane
import pennylane as qml

vqc = model.vqc
n, L = vqc.n, vqc.L

# 1) Module summary and weights
print(vqc)
print(f"\nRot weights {tuple(vqc.weights.shape)} = {vqc.weights.numel()} trainable params | "
      f"readout {vqc.out_features} features | Hilbert dim {2**n}\n")
for l in range(L):
    print(f"Layer {l}  (CNOT range r={vqc.ranges[l]})   rows = qubits, cols = (phi, theta, omega)")
    print(np.round(vqc.weights[l].detach().cpu().numpy(), 3))

# 2) Same circuit in PennyLane, for drawing
dev = qml.device("default.qubit", wires=n)

@qml.qnode(dev)
def vqc_circuit(a, weights):                      # a = encoded angles in (0, π)
    for l in range(L):
        if l == 0 or vqc.reupload:
            for i in range(n): qml.Hadamard(wires=i)
            for i in range(n): qml.RZ(a[i], wires=i)
            for i, j in vqc.pairs: qml.IsingZZ((np.pi - a[i]) * (np.pi - a[j]), wires=[i, j])
        for i in range(n): qml.Rot(*weights[l, i], wires=i)
        for i in range(n): qml.CNOT(wires=[i, (i + vqc.ranges[l]) % n])
    return ([qml.expval(qml.PauliZ(i)) for i in range(n)] +
            [qml.expval(qml.PauliZ(i) @ qml.PauliZ(j)) for i, j in vqc.pairs])

# 3) Real input: C5 spikes of one test digit at t=0 -> q_in -> angles
img, lbl = test_ds[0]
model.reset_state()
with torch.no_grad():
    l1, l2, l3, _ = model.lifs
    s = rate_encode(img.unsqueeze(0).to(device), 1, CFG["gain"])[0]
    s = F.max_pool2d(l1(model.conv1(s)), 2)
    s = F.max_pool2d(l2(model.conv2(s)), 2)
    s = l3(model.fc1(s.flatten(1)))
    x_q = model.q_in(s)                                       # (1, n) pre-activation
model.reset_state()
a = (math.pi * torch.sigmoid(x_q))[0].cpu().numpy()
weights = vqc.weights.detach().cpu().numpy()

# 4) Text drawing
print(f"\nCircuit for test digit (label {lbl}), t=0:\n")
print(qml.draw(vqc_circuit, decimals=2, max_length=250)(a, weights))

# 5) Figure
fig, ax = qml.draw_mpl(vqc_circuit, decimals=2, style="pennylane")(a, weights)
fig.suptitle(f"v2 VQC: ZZ feature map + strongly-entangling layers ({n} qubits × {L} layers)")
fig.savefig("vqc_layer_v2.png", dpi=150, bbox_inches="tight")
plt.show()

# 6) Sanity check
with torch.no_grad():
    ours = vqc(x_q)[0].cpu().numpy()
print("\n<Z>,<ZZ> model     :", np.round(ours, 4))
print("<Z>,<ZZ> PennyLane :", np.round(np.array(vqc_circuit(a, weights)), 4))

## 8. Classical spiking LeNet-5 baseline (for comparison)

In [ ]:
class SpikingLeNet5(nn.Module):
    LAYERS = ["C1", "C3", "C5", "F6", "F7-out"]
    def __init__(self, num_classes=10, beta=0.9, threshold=1.0):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2); self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(400, 120); self.fc2 = nn.Linear(120, 84); self.fc3 = nn.Linear(84, num_classes)
        self.lifs = nn.ModuleList([LIF(beta, threshold) for _ in range(5)])
    def reset_state(self):
        for l in self.lifs: l.reset_state()
    def forward(self, spikes_in, record=False):
        self.reset_state()
        l1, l2, l3, l4, lo = self.lifs
        counts, rates, out_train = 0.0, [0.0] * 5, []
        T = spikes_in.shape[0]
        for t in range(T):
            s1 = F.max_pool2d(l1(self.conv1(spikes_in[t])), 2)
            s2 = F.max_pool2d(l2(self.conv2(s1)), 2)
            s3 = l3(self.fc1(s2.flatten(1))); s4 = l4(self.fc2(s3)); so = lo(self.fc3(s4))
            counts = counts + so
            if record:
                out_train.append(so.detach())
                for i, s in enumerate((s1, s2, s3, s4, so)): rates[i] += s.detach().mean().item() / T
        return (counts, rates, torch.stack(out_train), None) if record else counts

### 8a. Width-matched classical control
The hybrid architecture with the VQC replaced by a classical layer with the same interface:
$n$ inputs → $\tanh$ → FC($n$→$2n$) → $\tanh$ → $2n$ outputs in $[-1,1]$ ($2n^2+2n$ params vs. $3nL$ for the VQC).
This isolates the contribution of the **quantum** layer from the effect of the **narrow bottleneck**.

In [ ]:
class ClassicalBottleneck(nn.Module):
    """Drop-in classical replacement for the VQC: n inputs -> 2n outputs in [-1,1]."""
    def __init__(self, n):
        super().__init__(); self.lin = nn.Linear(n, 2 * n); self.shots = None
    def forward(self, x):
        return torch.tanh(self.lin(torch.tanh(x)))

def make_classical_control():
    m = HybridQSpikingLeNet5(CFG["n_qubits"], CFG["q_layers"], CFG["reupload"], None,
                             beta=CFG["beta"], threshold=CFG["threshold"])
    m.vqc = ClassicalBottleneck(CFG["n_qubits"])
    return m.to(device)

## 9. Training and evaluation loops (shared by both models)

In [ ]:
def train_epoch(model, loader, opt):
    model.train()
    tot, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False, desc="train"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        counts = model(rate_encode(x, CFG["T"], CFG["gain"]))
        loss = F.cross_entropy(counts, y)            # spike counts as logits (rate decoding)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        tot += loss.item() * y.size(0); correct += (counts.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, 100.0 * correct / n


@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    model.eval()
    tot, correct, n, rate_sum, preds, targets = 0.0, 0, 0, None, [], []
    for x, y in tqdm(loader, leave=False, desc="eval"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        counts, rates, _, _ = model(rate_encode(x, CFG["T"], CFG["gain"]), record=True)
        tot += F.cross_entropy(counts, y).item() * y.size(0)
        p = counts.argmax(1); correct += (p == y).sum().item(); n += y.size(0)
        rate_sum = np.array(rates) if rate_sum is None else rate_sum + np.array(rates)
        if return_preds: preds.append(p.cpu()); targets.append(y.cpu())
    out = dict(loss=tot / n, acc=100.0 * correct / n, rates=rate_sum / len(loader))
    if return_preds:
        out["preds"], out["targets"] = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return out


def fit(model, opt, epochs, ckpt, tag):
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist, best = dict(train_loss=[], train_acc=[], test_loss=[], test_acc=[], rates=[], time=[]), 0.0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_dl, opt)
        te = evaluate(model, test_dl)
        sched.step()
        for k, v in zip(["train_loss", "train_acc", "test_loss", "test_acc", "rates", "time"],
                        [tr_loss, tr_acc, te["loss"], te["acc"], te["rates"], time.time() - t0]):
            hist[k].append(v)
        if te["acc"] > best:
            best = te["acc"]; torch.save(model.state_dict(), ckpt)
        print(f"[{tag}] Ep {ep:02d} | train {tr_loss:.4f} / {tr_acc:.2f}% | test {te['loss']:.4f} / {te['acc']:.2f}% "
              f"| best {best:.2f}% | rates [{', '.join(f'{r:.3f}' for r in te['rates'])}] | {hist['time'][-1]:.1f}s")
    print(f"[{tag}] best test accuracy: {best:.2f}%  -> {ckpt}")
    return hist, best

## 10. Train the hybrid model
Separate learning rates for classical weights and quantum circuit angles.

In [ ]:
q_params = list(model.vqc.parameters())
q_ids = {id(p) for p in q_params}
c_params = [p for p in model.parameters() if id(p) not in q_ids]
opt = torch.optim.Adam([{"params": c_params, "lr": CFG["lr"]},
                        {"params": q_params, "lr": CFG["q_lr"]}])
hist, best = fit(model, opt, CFG["epochs"], CFG["ckpt"], "Hybrid-Q")

## 11. Learning curves and firing rates

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1); rates = np.array(hist["rates"])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep, hist["train_loss"], "o-", label="train"); ax[0].plot(ep, hist["test_loss"], "s-", label="test")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["train_acc"], "o-", label="train"); ax[1].plot(ep, hist["test_acc"], "s-", label="test")
ax[1].set(title="Accuracy (%)", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
for i, name in enumerate(HybridQSpikingLeNet5.LAYERS): ax[2].plot(ep, rates[:, i], "o-", label=name)
ax[2].set(title="Mean firing rate (spikes/neuron/step)", xlabel="epoch"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 12. Final evaluation of best hybrid checkpoint

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt"], map_location=device))
res = evaluate(model, test_dl, return_preds=True)
print(f"Hybrid test accuracy: {res['acc']:.2f}%   loss: {res['loss']:.4f}")
print("Firing rates:", dict(zip(HybridQSpikingLeNet5.LAYERS, np.round(res["rates"], 4))))
print(); print(classification_report(res["targets"], res["preds"], digits=4))

cm = confusion_matrix(res["targets"], res["preds"])
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="predicted", ylabel="true", title="Confusion matrix (hybrid)")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

## 13. Inside the quantum layer: ⟨Z⟩/⟨ZZ⟩ traces and output spike rasters

In [ ]:
x, y = next(iter(test_dl)); x, y = x[:4].to(device), y[:4]
with torch.no_grad():
    counts, _, out_train, q_trace = model(rate_encode(x, CFG["T"], CFG["gain"]), record=True)

fig, ax = plt.subplots(3, 4, figsize=(16, 8), gridspec_kw=dict(height_ratios=[1, 1, 1.1]))
for k in range(4):
    ax[0, k].imshow(x[k, 0].cpu(), cmap="gray"); ax[0, k].axis("off")
    ax[0, k].set_title(f"true {y[k].item()} | pred {counts[k].argmax().item()}")
    im = ax[1, k].imshow(q_trace[:, k].cpu().T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
    ax[1, k].axhline(CFG["n_qubits"] - .5, c="k", lw=.8)
    ax[1, k].set(xlabel="time step", ylabel="feature (⟨Z⟩ | ⟨ZZ⟩)", title="quantum features per step")
    tt, nn_ = np.nonzero(out_train[:, k].cpu().numpy())
    ax[2, k].scatter(tt, nn_, marker="|", s=80, c="k")
    ax[2, k].axhline(y[k].item(), color="tab:green", alpha=.25, lw=8)
    ax[2, k].set(xlim=(-0.5, CFG["T"] - .5), ylim=(-.5, 9.5), yticks=range(10), xlabel="time step", ylabel="output neuron")
fig.colorbar(im, ax=ax[1, :].tolist(), fraction=.015)
plt.suptitle("Input · quantum-layer expectation values · output spike raster (green = true class)")
plt.show()

# class-conditional mean <Z> (time-averaged) over a test subset
zs, ys = [], []
with torch.no_grad():
    for i, (xb, yb) in enumerate(test_dl):
        if i == 8: break
        _, _, _, qt = model(rate_encode(xb.to(device), CFG["T"], CFG["gain"]), record=True)
        zs.append(qt.mean(0).cpu()); ys.append(yb)
zs, ys = torch.cat(zs), torch.cat(ys)
cls_mean = torch.stack([zs[ys == c].mean(0) for c in range(10)])
plt.figure(figsize=(6, 4)); plt.imshow(cls_mean, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="mean expectation"); plt.axvline(CFG["n_qubits"] - .5, c="k", lw=.8); plt.xlabel("feature: ⟨Z_i⟩ | ⟨Z_iZ_i+1⟩"); plt.ylabel("class"); plt.yticks(range(10))
plt.title("Class-conditional quantum feature map"); plt.tight_layout(); plt.show()

## 14. Robustness: inference time steps and finite measurement shots

In [ ]:
T_train = CFG["T"]
T_list = [1, 2, 5, 10, 15, 20, 25, 30, 40]
acc_T = []
for T in T_list:
    CFG["T"] = T; acc_T.append(evaluate(model, test_dl)["acc"]); print(f"T={T:3d}  acc={acc_T[-1]:.2f}%")
CFG["T"] = T_train

shot_list = [None, 4096, 1024, 256, 64, 16]
acc_S = []
for s in shot_list:
    model.vqc.shots = s; acc_S.append(evaluate(model, test_dl)["acc"])
    print(f"shots={str(s):>5}  acc={acc_S[-1]:.2f}%")
model.vqc.shots = CFG["shots"]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(T_list, acc_T, "o-"); ax[0].axvline(T_train, ls="--", c="gray", label=f"T_train={T_train}")
ax[0].set(xlabel="inference time steps T", ylabel="test accuracy (%)", title="Accuracy vs latency"); ax[0].legend(); ax[0].grid(alpha=.3)
lbls = ["exact" if s is None else str(s) for s in shot_list]
ax[1].plot(range(len(shot_list)), acc_S, "o-"); ax[1].set_xticks(range(len(shot_list))); ax[1].set_xticklabels(lbls)
ax[1].set(xlabel="shots per ⟨Z⟩ estimate", ylabel="test accuracy (%)", title="Accuracy vs measurement shots"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 15. Baselines and comparison
* **Classical SNN**: standard spiking LeNet-5 (F6 = FC 120→84).
* **Classical control**: the hybrid architecture with the VQC replaced by a width-matched classical layer (section 8a).

The hybrid-vs-control gap measures the contribution of the quantum layer itself. Set `CFG['run_baseline']=False` to skip.

In [ ]:
if CFG["run_baseline"]:
    results = {"Hybrid Q-SNN v2": dict(hist=hist, best=best,
                                    params=sum(p.numel() for p in model.parameters()),
                                    f6=f"ZZ-SEL {CFG['n_qubits']}q×{CFG['q_layers']}L")}

    set_seed(CFG["seed"])
    base = SpikingLeNet5(beta=CFG["beta"], threshold=CFG["threshold"]).to(device)
    h, b = fit(base, torch.optim.Adam(base.parameters(), lr=CFG["lr"]), CFG["epochs"], CFG["baseline_ckpt"], "Classical SNN")
    results["Classical SNN"] = dict(hist=h, best=b, params=sum(p.numel() for p in base.parameters()), f6="FC 120→84")

    set_seed(CFG["seed"])
    ctrl = make_classical_control()
    h, b = fit(ctrl, torch.optim.Adam(ctrl.parameters(), lr=CFG["lr"]), CFG["epochs"],
               CFG["baseline_ckpt"].replace(".pt", "_widthmatched_v2.pt"), "Classical ctrl")
    results["Classical control"] = dict(hist=h, best=b, params=sum(p.numel() for p in ctrl.parameters()),
                                        f6=f"tanh-FC {CFG['n_qubits']}→{2*CFG['n_qubits']}")

    names = ["Classical SNN", "Classical control", "Hybrid Q-SNN v2"]
    row = lambda label, vals, fmt: print(f"{label:<24}" + "".join(f"{v:>20{fmt}}" for v in vals))
    print(); row("", names, "")
    row("F6 layer", [results[k]["f6"] for k in names], "")
    row("trainable params", [results[k]["params"] for k in names], ",")
    row("best test acc (%)", [results[k]["best"] for k in names], ".2f")
    row("mean epoch time (s)", [np.mean(results[k]["hist"]["time"]) for k in names], ".1f")

    ep = np.arange(1, CFG["epochs"] + 1)
    plt.figure(figsize=(7, 4.2))
    for k, mk in zip(names, ["o-", "^-", "s-"]):
        plt.plot(ep, results[k]["hist"]["test_acc"], mk, label=k)
    plt.xlabel("epoch"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3); plt.legend()
    plt.title("Test accuracy per epoch"); plt.tight_layout(); plt.show()